In [1]:
# import required modules 
    # copied these over from part 1
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from pathlib import Path
import anndata
import numpy as np
import os
import scanpy as sc
import pandas as pd
import time
import matplotlib.pyplot as plt

In [2]:
# assigning our own broad cluster and subcluster hierarchy

# LOAD IN DOWNSAMPLED, MERGED OBJECT WITH CLUSTER METADATA ATTACHED 
adata = anndata.read_h5ad("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260123_WMB_10xv3_downsampled_metadata.h5ad")
adata



# list clusters from object metadata
adata.obs["cluster"]

# create empty metadata columns 
adata.obs["Broad_Cluster"] = pd.NA
adata.obs["Subcluster"] = pd.NA

# CREATE DICTIONARY FOR BROAD CLUSTERS (how you want the broad clusters organized)

Broad_Cluster_map_by_supertype = {
    "NK Cells" : ["1200 NK cells NN_3"], 
    "T Cells" : ["1201 T cells NN_4"]
}


Broad_Cluster_map_by_subclass = {
    "Microglia" : ["334 Microglia NN"], 
    "BAM" : ["335 BAM NN"], 
    "Monocytes" : ["336 Monocytes NN"], 
    "DC" : ["337 DC NN"], 
    "ABC" : ["329 ABC NN"], 
    "VLMC" : ["330 VLMC NN"], 
    "Pericytes" : ["331 Peri NN"], 
    "SMC" : ["332 SMC NN"], 
    "Endothelial Cells" : ["333 Endo NN"], 
    "OPC" : ["326 OPC NN"], 
    "Oligodendrocytes" : ["327 Oligo NN"], 
    "Astrocytes" : ["317 Astro-CB NN", "318 Astro-NT NN", "319 Astro-TE NN", "320 Astro-OLF NN"], 
    "Astroependymal" : ["321 Astroependymal NN"], 
    "Tanycytes" : ["322 Tanycyte NN"], 
    "Ependymal" : ["323 Ependymal NN"], 
    "Hypendymal" : ["324 Hypendymal NN"], 
    "CP" : ["325 CHOR NN"]
}


Broad_Cluster_map_by_class = {
    "Dopaminergic_Neurons" : ["21 MB Dopa"],
    "Serotonergic_Neurons" : ["22 MB-HB Sero"],
    "GABAergic_Neurons" : ["05 OB-IMN GABA", "06 CTX-CGE GABA", "07 CTX-MGE GABA", "08 CNU-MGE GABA", "09 CNU-LGE GABA", "10 LSX GABA", "11 CNU-HYa GABA", "12 HY GABA", "20 MB GABA", "26 P GABA", "27 MY GABA", "28 CB GABA"],
    "Glutamatergic_Neurons" : ["01 IT-ET Glut", "02 NP-CT-L6b Glut", "03 OB-CR Glut", "04 DG-IMN Glut", "13 CNU-HYa Glut", "14 HY Glut", "15 HY Gnrh1 Glut", "16 HY MM Glut", "17 MH-LH Glut", "18 TH Glut", "19 MB Glut", "23 P Glut", "24 MY Glut", "25 Pineal Glut", "29 CB Glut"]
}


# CREATE DICTIONARY FOR SUBCLUSTERS 


Subcluster_map_by_supertype = {
    "NK Cells" : ["1200 NK cells NN_3"], 
    "T Cells" : ["1201 T cells NN_4"]
}

Subcluster_map_by_subclass = {
    "Microglia" : ["334 Microglia NN"], 
    "BAM" : ["335 BAM NN"], 
    "Monocytes" : ["336 Monocytes NN"], 
    "DC" : ["337 DC NN"], 
    "ABC" : ["329 ABC NN"], 
    "VLMC" : ["330 VLMC NN"], 
    "Pericytes" : ["331 Peri NN"], 
    "SMC" : ["332 SMC NN"], 
    "Endothelial Cells" : ["333 Endo NN"], 
    "OPC" : ["326 OPC NN"], 
    "Oligodendrocytes" : ["327 Oligo NN"], 
    "Astrocytes" : ["317 Astro-CB NN", "318 Astro-NT NN", "319 Astro-TE NN", "320 Astro-OLF NN"], 
    "Astroependymal" : ["321 Astroependymal NN"], 
    "Tanycytes" : ["322 Tanycyte NN"], 
    "Ependymal" : ["323 Ependymal NN"], 
    "Hypendymal" : ["324 Hypendymal NN"], 
    "CP" : ["325 CHOR NN"]
}

Subcluster_map_by_class = {
    "Dopaminergic_Neurons" : ["21 MB Dopa"],
    "Serotonergic_Neurons" : ["22 MB-HB Sero"]
}

GABA_Subcluster_map_by_subclass = {
    "LAMP5" : ["049 Lamp5 Gaba", "050 Lamp5 Lhx6 Gaba"], 
    "PVALB" : ["051 Pvalb chandelier Gaba", "052 Pvalb Gaba"], 
    "SNCG" : ["047 Sncg Gaba"], 
    "SST" : ["053 Sst Gaba", "056 Sst Chodl Gaba"], 
    "VIP" : ["046 Vip Gaba"]
}

Glut_Subcluster_map_by_subclass = {
    "L2_3_IT" : ["007 L2/3 IT CTX Glut", "008 L2/3 IT ENT Glut", "009 L2/3 IT PIR-ENTl Glut", "019 L2/3 IT PPP Glut", "020 L2/3 IT RSP Glut"], 
    "L5_IT" : ["005 L5 IT CTX Glut"], 
    "L6_IT" : ["004 L6 IT CTX Glut"], 
    "L5_ET" : ["022 L5 ET CTX Glut"], 
    "L6_CT" : ["028 L6b/CT ENT Glut", "029 L6b CTX Glut", "030 L6 CT CTX Glut"]
}


    


In [3]:
# adding map to object 

# extracting metadata from downsampled object 
meta = adata.obs.copy()

'''
Function: used to map predefined "dictionaries" from above to the downsampled 10xv3 object metadata
Input: metadata row (must be a pandas df) that has ABC cluster metadata informatoin attached 
    (contains columns 'class', 'subclass', and 'cluster'"
Output: labels the pre-defined broad cluster that the cell belongs to in the 'Broad_Cluster' column
'''

def map_broad_cluster(row):
    
    # Mapping by class first 
    # .items() returns broad_cluster (broad) and class (class_list) pairs from the 'Broad_Cluster_map_by_class' df
    for broad, class_list in Broad_Cluster_map_by_class.items():

        # for loop goes through each class in the class_list belonging to the broad_cluster value
        for cls in class_list:

            # if the class matches the class listed in the 'class' column from the downsampled object, the 'broad_cluster' value will be added to the 'Broad_Cluster' column
            if cls in row['class']:
                return broad
    
    # if couldn't find a match for class, then use Broad_Cluster_map_by_subclass
    for broad, subclass_list in Broad_Cluster_map_by_subclass.items():

        # for loop goes through each subclass in the subclass_list belonging to the broad_cluster value
        for subcls in subclass_list:
            
            # if the subclass matches, the 'broad_cluster' value will be added 
            if subcls in row['subclass']:
                return broad

    # if couldn't find a match for subclass, then use Broad_Cluster_map_by_supertype
    for broad, supertype_list in Broad_Cluster_map_by_supertype.items():

        # for loop goes through each supertype in the supertype_list belonging to the broad_cluster value
        for suptype in supertype_list:
            
            # if the supertype matches, the 'broad_cluster' value will be added 
            if suptype in row['supertype']:
                return broad
                
    
    # if nothing matches
    return "Unknown"

def map_subcluster(row):

    # for neuronal subclusters 
    if row['Broad_Cluster'] == "Glutamatergic_Neurons":
        for sub, subclass_list in Glut_Subcluster_map_by_subclass.items():
            for subcls in subclass_list:
                if subcls in row['subclass']:
                    return sub
                    
        return None 
    
    if row['Broad_Cluster'] == "GABAergic_Neurons":
        for sub, subclass_list in GABA_Subcluster_map_by_subclass.items():
            for subcls in subclass_list:
                if subcls in row['subclass']:
                    return sub
                    
        return None    

    
    # other subclusters  

    # mapping by class dictionary
    for sub, class_list in Subcluster_map_by_class.items():
        for cls in class_list:
            if cls in row['class']:
                return sub
                
    # mapping by subclass dictionary 
    for sub, subclass_list in Subcluster_map_by_subclass.items():
        for s in subclass_list:
            if s in row['subclass']:
                return sub
                
    # mapping by supertype dictionary  
    for sub, supertype_list in Subcluster_map_by_supertype.items():
        for su in supertype_list:
            if su in row['supertype']:
                return sub

    
    return None




# applying the function to the metadata object 
meta['Broad_Cluster'] = meta.apply(map_broad_cluster, axis = 1)
meta['Subcluster'] = meta.apply(map_subcluster, axis = 1)

# dropping cells that were not assigned to a broad cluster 
'''
In this case, there are no unannotated cells present (we removed them in part 2). 
The cells in the unknown cluster are cells belonging to classes that we did not specify in the dictionary 
bc we do not need them for ARIA project (i.e. OEC, B cells, etc.) and we don't have any markers for them in our gene panel.
'''
meta = meta[meta['Broad_Cluster'] != "Unknown"].copy()

# assign back to the downsampled obj
adata = adata[meta.index, :].copy()
adata.obs = meta



# save downsampled object with cluster metadata and our own cluster levels added in
adata.write_h5ad("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260123_WMB_10xv3_FINAL.h5ad")



In [4]:
print(adata.n_obs)
print(adata.obs["Broad_Cluster"].value_counts())

203880
Broad_Cluster
Oligodendrocytes         58259
Astrocytes               43877
Endothelial Cells        16174
OPC                      14814
Glutamatergic_Neurons    12783
GABAergic_Neurons        12000
SMC                       9009
Microglia                 8554
VLMC                      8544
Pericytes                 5611
BAM                       5516
Ependymal                 3096
Astroependymal            1174
Tanycytes                 1004
Serotonergic_Neurons      1000
ABC                        824
Dopaminergic_Neurons       500
CP                         447
DC                         261
T Cells                    165
NK Cells                   128
Hypendymal                 111
Monocytes                   29
Name: count, dtype: int64
